# Notebook 04: Grouped search and out-of-fold scores (2023 to 2024)

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook implements the tuning and development comparison in sections 3.7.2 and 3.7.3, up to the out-of-fold scores.

**What it does**
1. **Phase 1, tuning.** For each model (logistic regression, random forest, histogram gradient boosting) it fits every configuration in the budget on the untreated training data, across the five saved collision-grouped folds, and scores each on the held-out fold by average precision.
2. Picks the configuration with the highest mean fold average precision for each model.
3. **Phase 2, training treatments.** Refits the selected settings with inverse-frequency weighting and with random oversampling, on the same five folds.
4. Saves the out-of-fold scores for all nine model and training combinations, and per-fold metrics.

**Round 2 (optional, `ROUND2 = True`).** If a model's best setting sits at the edge of its grid and compute is left over, a short second round adds configurations beyond that edge. The round-1 results and selection are kept unchanged on disk, so both rounds can be reported.

**Logistic regression plateau rule.** If several penalty strengths score within 0.0005 average precision of the best, the strongest penalty (smallest C) is used, since the fitted terms are easier to interpret and the scores are equivalent.

**Safe to interrupt.** Every fold is saved to Drive the moment it finishes. If Colab disconnects, reopen the notebook, run the setup cells and rerun. Finished folds are skipped.

**What it does not do:** no thresholds, no model choice, no 2025 data. Thresholds and the one-standard-error rule are applied in notebook 05 on the saved scores.

**Dry run:** set `QUICK_TEST = True` in the setup cell to check that everything works on the pilot rows in a few minutes. It saves to a separate folder and does not touch the real results.

In [ ]:
# 1. Setup -----------------------------------------------------------------
import os, sys, json, hashlib, subprocess, time, gc, warnings, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
try:
    import imblearn, psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'psutil'], check=True)
    import imblearn, psutil
try:
    display
except NameError:
    display = print

QUICK_TEST = False          # True = dry run on the pilot rows, 2 settings per model, separate output folder
ROUND2 = True               # add the second-round configurations defined in section 2 (set False to reproduce round 1 only)
MAX_RUN_MINUTES = 0         # 0 = run until finished. Set e.g. 120 to stop cleanly after that long, then rerun to continue.

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))
CODE = PROJECT / 'code'; MOD = PROJECT / 'outputs' / 'models'; ENV = PROJECT / 'environment'
SEARCH = MOD / ('search_quicktest' if QUICK_TEST else 'search')
SEARCH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
from stats19_lib import *
import stats19_lib

SEED = 20261107
N_JOBS = os.cpu_count()
T_START = time.time()
(ENV / 'pip_freeze_notebook04.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
print('scikit-learn', sklearn.__version__, '| cores', N_JOBS, '| shared code sha256', hashlib.sha256((CODE / 'stats19_lib.py').read_bytes()).hexdigest()[:16])
print('Output folder:', SEARCH)

Mounted at /content/drive
scikit-learn 1.6.1 | cores 2 | shared code sha256 ab3f7748e0fc5fd1
Output folder: /content/drive/MyDrive/stats19_severity/outputs/models/search


In [ ]:
# 2. Data, folds and the fixed list of configurations ---------------------------------
dev, NUM, CAT = load_dev(PROJECT)
sha = hashlib.sha256((PROJECT / 'data' / 'model_inputs' / 'folds_dev.csv').read_bytes()).hexdigest()
assert sha == json.loads((MOD / 'folds_checksum.json').read_text())['sha256'], 'folds_dev.csv has changed since notebook 03'
budget = json.loads((MOD / 'search_budget.json').read_text())
assert budget['sklearn'] == sklearn.__version__, f"scikit-learn changed since notebook 03 ({budget['sklearn']} -> {sklearn.__version__}). Rerun notebook 03."
if QUICK_TEST:
    dev = dev[dev['pilot'] == 1].reset_index(drop=True)
dev = dev.reset_index(drop=True)
folds = sorted(dev['fold'].unique())
print('Rows:', len(dev), '| folds:', folds, '| KSI rate', round(dev['ksi'].mean(), 4))

n_cfg = {k: (min(v, 2) if QUICK_TEST else v) for k, v in budget['configs_per_model'].items()}
CONFIGS = {k: sample_configs(k, n_cfg[k], SEED) for k in KINDS}
cfg_path = SEARCH / 'search_configs.json'          # round 1 (as set by the budget); never changed once saved
if cfg_path.exists():
    assert json.loads(cfg_path.read_text()) == json.loads(json.dumps(CONFIGS)), 'Round-1 configuration list differs from the saved one.'
else:
    cfg_path.write_text(json.dumps(CONFIGS, indent=1))
N_BASE = {k: len(CONFIGS[k]) for k in KINDS}

# Round 2: explicit configurations placed beyond the edges that round 1 reached (best random forest setting at the largest
# max_features tried; best boosting setting at the largest leaf count and l2 of 0; logistic regression flat at the top of its grid).
EXTRA = {
    'lr':  [dict(C=3.0), dict(C=10.0)],
    'rf':  [dict(max_features=0.3, min_samples_leaf=10, min_samples_split=10), dict(max_features=0.5, min_samples_leaf=10, min_samples_split=10)],
    'hgb': [dict(learning_rate=0.05, max_iter=200, max_leaf_nodes=127, min_samples_leaf=50, l2_regularization=0.0),
            dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=50, l2_regularization=0.0)],
}
# The first round-2 results showed the forest's best max_features is interior (0.2 beat 0.3 and 0.5), so the other
# planned forest and boosting extras were dropped. Configurations already run keep their numbers.
if ROUND2:
    for k in KINDS:
        ex = EXTRA[k][:1] if QUICK_TEST else EXTRA[k]
        assert all(e not in CONFIGS[k] for e in ex), 'a round-2 configuration duplicates a round-1 one'
        CONFIGS[k] = CONFIGS[k] + ex
    r2 = SEARCH / 'search_configs_round2.json'
    if r2.exists():
        saved = json.loads(r2.read_text()); cur = json.loads(json.dumps(CONFIGS))
        for k in KINDS:
            m = min(len(saved[k]), len(cur[k]))
            assert saved[k][:m] == cur[k][:m], f'{k}: configurations already run differ from the saved list'
    r2.write_text(json.dumps(CONFIGS, indent=1))
for k in KINDS:
    print(f'\n{k}: {len(CONFIGS[k])} configurations ({N_BASE[k]} round 1' + (f', {len(CONFIGS[k]) - N_BASE[k]} round 2)' if ROUND2 else ')'))
    display(pd.DataFrame(CONFIGS[k]))
total_fits = sum(len(CONFIGS[k]) * len(folds) for k in KINDS) + len(KINDS) * 2 * len(folds)
print('\nTotal fits planned:', total_fits)

Rows: 261249 | folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | KSI rate 0.2117

lr: 9 configurations (7 round 1, 2 round 2)


,C
0,0.001
1,0.003
2,0.010
3,0.030
4,0.100
5,0.300
6,1.000
7,3.000
8,10.000



rf: 6 configurations (4 round 1, 2 round 2)


,max_features,min_samples_leaf,min_samples_split
0,sqrt,20,20
1,0.2,10,10
2,0.2,50,50
3,0.2,50,10
4,0.3,10,10
5,0.5,10,10



hgb: 8 configurations (6 round 1, 2 round 2)


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.10,200,31,50,1.0
1,0.03,200,63,20,1.0
2,0.05,200,63,50,0.0
3,0.10,100,31,20,1.0
4,0.03,100,15,100,1.0
5,0.05,300,15,100,0.0
6,0.05,200,127,50,0.0
7,0.05,300,63,50,0.0



Total fits planned: 145


In [ ]:
# 3. Helpers: one fold, saved immediately -------------------------------------------
LOG = SEARCH / 'search_log.csv'
done_count = 0
class OutOfTime(Exception): pass

def fold_file(kind, trt, cfg, k):
    return SEARCH / f'{kind}__{trt}__cfg{cfg}__fold{k}.npy'

def run_fold(kind, trt, cfg_id, params, k):
    global done_count
    f = fold_file(kind, trt, cfg_id, k)
    if f.exists():
        return np.load(f)
    if MAX_RUN_MINUTES and (time.time() - T_START) / 60 > MAX_RUN_MINUTES:
        raise OutOfTime()
    tr = dev[dev['fold'] != k]; va = dev[dev['fold'] == k]
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        p, info = fit_predict(kind, params, trt, tr, tr['ksi'], va, NUM, CAT, SEED, N_JOBS)
    n_conv = sum(issubclass(x.category, ConvergenceWarning) for x in w)
    tmp = f.with_suffix('.tmp.npy'); np.save(tmp, p.astype(np.float32)); tmp.replace(f)
    m = metrics(va['ksi'], p)
    row = dict(kind=kind, treatment=trt, cfg=cfg_id, fold=k, ap=round(m['ap'], 5), t_total_min=round((info['t_prep'] + info['t_fit'] + info['t_pred']) / 60, 2),
               n_train=info['n_train'], conv_warnings=n_conv, params=json.dumps(params), finished=dt.datetime.now().isoformat(timespec='seconds'))
    pd.DataFrame([row]).to_csv(LOG, mode='a', header=not LOG.exists(), index=False)
    done_count += 1
    print(f"{kind:>3} {trt:<10} cfg {cfg_id} fold {k} | AP {m['ap']:.4f} | {row['t_total_min']:.1f} min | conv warn {n_conv} | elapsed {(time.time()-T_START)/3600:.2f} h")
    del tr, va; gc.collect()
    return p

# Compare actual with projected time on the first full-size fit of each model (anchor configuration).
pt = pd.read_csv(MOD / 'pilot_timing.csv').set_index(['model', 'training'])['t_full_fit_min'] if (MOD / 'pilot_timing.csv').exists() else None

In [ ]:
# 4. Phase 1: tune each model on the untreated fit ----------------------------------------
# Order: logistic regression, random forest, boosting. Each fold is skipped if already saved.
try:
    for kind in KINDS:
        for ci, params in enumerate(CONFIGS[kind]):
            for k in folds:
                run_fold(kind, 'untreated', ci, params, k)
            if ci == 0 and pt is not None and not QUICK_TEST:
                row = pd.read_csv(LOG).query("kind == @kind and cfg == 0 and treatment == 'untreated'")
                print(f"   -> {kind} anchor: actual {row['t_total_min'].mean():.2f} min per fold vs projected {pt[(kind, 'untreated')] * 1.0:.2f} min")
    print('\nPhase 1 complete.')
except OutOfTime:
    print(f'\nStopped after {MAX_RUN_MINUTES} minutes as requested. Rerun this cell to continue; finished folds are kept.')
    raise SystemExit

   -> lr anchor: actual 0.11 min per fold vs projected 0.25 min
   -> rf anchor: actual 1.14 min per fold vs projected 1.28 min
   -> hgb anchor: actual 0.50 min per fold vs projected 3.16 min
hgb untreated  cfg 6 fold 0 | AP 0.4446 | 3.5 min | conv warn 0 | elapsed 0.08 h
hgb untreated  cfg 6 fold 1 | AP 0.4575 | 3.5 min | conv warn 0 | elapsed 0.14 h
hgb untreated  cfg 6 fold 2 | AP 0.4612 | 3.5 min | conv warn 0 | elapsed 0.20 h
hgb untreated  cfg 6 fold 3 | AP 0.4507 | 3.5 min | conv warn 0 | elapsed 0.25 h
hgb untreated  cfg 6 fold 4 | AP 0.4638 | 3.5 min | conv warn 0 | elapsed 0.31 h
hgb untreated  cfg 7 fold 0 | AP 0.4456 | 3.8 min | conv warn 0 | elapsed 0.38 h
hgb untreated  cfg 7 fold 1 | AP 0.4561 | 3.8 min | conv warn 0 | elapsed 0.44 h
hgb untreated  cfg 7 fold 2 | AP 0.4616 | 3.8 min | conv warn 0 | elapsed 0.50 h
hgb untreated  cfg 7 fold 3 | AP 0.4490 | 3.8 min | conv warn 0 | elapsed 0.57 h
hgb untreated  cfg 7 fold 4 | AP 0.4655 | 3.8 min | conv warn 0 | elapsed 0.63

In [ ]:
# 5. Select the settings with the highest mean fold average precision ---------------------
y = dev['ksi'].values; fold_arr = dev['fold'].values
def fold_scores(kind, trt, cfg):
    out = []
    for k in folds:
        p = np.load(fold_file(kind, trt, cfg, k)); m = fold_arr == k
        out.append(average_precision_score(y[m], p))
    return np.array(out)

rows = []
for kind in KINDS:
    for ci, params in enumerate(CONFIGS[kind]):
        s = fold_scores(kind, 'untreated', ci)
        rows.append(dict(model=kind, cfg=ci, mean_ap=s.mean(), sd_ap=s.std(ddof=1), se_ap=s.std(ddof=1) / np.sqrt(len(s)), **{f'p_{a}': b for a, b in params.items()}))
summary = pd.DataFrame(rows)
summary.insert(2, 'round', np.where(summary['cfg'] < summary['model'].map(N_BASE), 1, 2))
summary.round(4).to_csv(SEARCH / 'search_summary.csv', index=False)
BEST = {}
for kind in KINDS:
    sub = summary[summary['model'] == kind].sort_values(['mean_ap', 'cfg'], ascending=[False, True])
    BEST[kind] = int(sub.iloc[0]['cfg'])
    if kind == 'lr':      # flat plateau: settings within 0.0005 of the best are equivalent, so take the strongest penalty (smallest C)
        near = sub[sub['mean_ap'] >= sub['mean_ap'].max() - 0.0005]
        BEST[kind] = int(near.sort_values('p_C').iloc[0]['cfg'])
        if BEST[kind] != int(sub.iloc[0]['cfg']):
            print('lr: plateau rule applied (settings within 0.0005 AP are equivalent): using the smallest C among them.')
    print(f"\n{kind}: selected configuration {BEST[kind]} (top mean AP {sub.iloc[0]['mean_ap']:.4f} +/- {sub.iloc[0]['se_ap']:.4f})")
    display(sub.round(4).reset_index(drop=True))
# A best setting at the edge of its grid means the grid may be too narrow; report it rather than hide it.
for kind in KINDS:
    b = CONFIGS[kind][BEST[kind]]
    at_edge = [name for name, v in b.items() if name in SPACE[kind] and v in (SPACE[kind][name][0], SPACE[kind][name][-1]) and len(SPACE[kind][name]) > 2]
    print(f'{kind}: best settings at the edge of the grid for: {at_edge or "none"}')
sel_path = SEARCH / 'selected_settings.json'
if sel_path.exists() and not (SEARCH / 'selected_settings_round1.json').exists() and ROUND2:
    (SEARCH / 'selected_settings_round1.json').write_text(sel_path.read_text())   # round-1 selection kept as a record
sel_path.write_text(json.dumps({k: dict(cfg=BEST[k], params=CONFIGS[k][BEST[k]]) for k in KINDS}, indent=1))

lr: plateau rule applied (settings within 0.0005 AP are equivalent): using the smallest C among them.

lr: selected configuration 4 (top mean AP 0.4349 +/- 0.0048)


,model,cfg,round,mean_ap,sd_ap,se_ap,p_C,p_max_features,p_min_samples_leaf,p_min_samples_split,p_learning_rate,p_max_iter,p_max_leaf_nodes,p_l2_regularization
0,lr,8,2,0.4349,0.0108,0.0048,10.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lr,6,1,0.4349,0.0109,0.0049,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lr,7,2,0.4349,0.0109,0.0049,3.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lr,5,1,0.4349,0.0111,0.0049,0.300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,lr,4,1,0.4348,0.0110,0.0049,0.100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,lr,3,1,0.4343,0.0104,0.0046,0.030,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lr,2,1,0.4329,0.0097,0.0044,0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lr,1,1,0.4292,0.0091,0.0040,0.003,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lr,0,1,0.4235,0.0085,0.0038,0.001,NaN,NaN,NaN,NaN,NaN,NaN,NaN



rf: selected configuration 1 (top mean AP 0.4486 +/- 0.0035)


,model,cfg,round,mean_ap,sd_ap,se_ap,p_C,p_max_features,p_min_samples_leaf,p_min_samples_split,p_learning_rate,p_max_iter,p_max_leaf_nodes,p_l2_regularization
0,rf,1,1,0.4486,0.0078,0.0035,NaN,0.2,10.0,10.0,NaN,NaN,NaN,NaN
1,rf,4,2,0.4477,0.0070,0.0031,NaN,0.3,10.0,10.0,NaN,NaN,NaN,NaN
2,rf,5,2,0.4457,0.0065,0.0029,NaN,0.5,10.0,10.0,NaN,NaN,NaN,NaN
3,rf,2,1,0.4455,0.0087,0.0039,NaN,0.2,50.0,50.0,NaN,NaN,NaN,NaN
4,rf,3,1,0.4455,0.0087,0.0039,NaN,0.2,50.0,10.0,NaN,NaN,NaN,NaN
5,rf,0,1,0.4438,0.0097,0.0043,NaN,sqrt,20.0,20.0,NaN,NaN,NaN,NaN



hgb: selected configuration 2 (top mean AP 0.4558 +/- 0.0037)


,model,cfg,round,mean_ap,sd_ap,se_ap,p_C,p_max_features,p_min_samples_leaf,p_min_samples_split,p_learning_rate,p_max_iter,p_max_leaf_nodes,p_l2_regularization
0,hgb,2,1,0.4558,0.0083,0.0037,NaN,NaN,50.0,NaN,0.05,200.0,63.0,0.0
1,hgb,7,2,0.4556,0.0083,0.0037,NaN,NaN,50.0,NaN,0.05,300.0,63.0,0.0
2,hgb,6,2,0.4556,0.0079,0.0035,NaN,NaN,50.0,NaN,0.05,200.0,127.0,0.0
3,hgb,1,1,0.4551,0.0086,0.0039,NaN,NaN,20.0,NaN,0.03,200.0,63.0,1.0
4,hgb,0,1,0.4539,0.0084,0.0038,NaN,NaN,50.0,NaN,0.10,200.0,31.0,1.0
5,hgb,3,1,0.4531,0.0086,0.0039,NaN,NaN,20.0,NaN,0.10,100.0,31.0,1.0
6,hgb,5,1,0.4510,0.0088,0.0039,NaN,NaN,100.0,NaN,0.05,300.0,15.0,0.0
7,hgb,4,1,0.4292,0.0083,0.0037,NaN,NaN,100.0,NaN,0.03,100.0,15.0,1.0


lr: best settings at the edge of the grid for: none
rf: best settings at the edge of the grid for: ['max_features', 'min_samples_split']
hgb: best settings at the edge of the grid for: ['max_leaf_nodes', 'l2_regularization']


348

In [ ]:
# 6. Phase 2: refit the selected settings under weighting and oversampling ------------------
try:
    for kind in KINDS:
        params = CONFIGS[kind][BEST[kind]]
        for trt in ('weight', 'oversample'):
            for k in folds:
                run_fold(kind, trt, BEST[kind], params, k)
    print('\nPhase 2 complete.')
except OutOfTime:
    print(f'\nStopped after {MAX_RUN_MINUTES} minutes as requested. Rerun this cell to continue.')
    raise SystemExit

 lr weight     cfg 4 fold 0 | AP 0.4173 | 0.2 min | conv warn 0 | elapsed 0.70 h
 lr weight     cfg 4 fold 1 | AP 0.4354 | 0.2 min | conv warn 0 | elapsed 0.70 h
 lr weight     cfg 4 fold 2 | AP 0.4422 | 0.2 min | conv warn 0 | elapsed 0.71 h
 lr weight     cfg 4 fold 3 | AP 0.4260 | 0.2 min | conv warn 0 | elapsed 0.71 h
 lr weight     cfg 4 fold 4 | AP 0.4431 | 0.2 min | conv warn 0 | elapsed 0.71 h
 lr oversample cfg 4 fold 0 | AP 0.4171 | 0.4 min | conv warn 0 | elapsed 0.72 h
 lr oversample cfg 4 fold 1 | AP 0.4353 | 0.3 min | conv warn 0 | elapsed 0.72 h
 lr oversample cfg 4 fold 2 | AP 0.4417 | 0.3 min | conv warn 0 | elapsed 0.73 h
 lr oversample cfg 4 fold 3 | AP 0.4257 | 0.3 min | conv warn 0 | elapsed 0.73 h
 lr oversample cfg 4 fold 4 | AP 0.4423 | 0.3 min | conv warn 0 | elapsed 0.74 h

Phase 2 complete.


In [ ]:
# 7. Assemble the out-of-fold scores and per-fold metrics -------------------------------------
oof = dev[['cas_key', 'fold', 'ksi']].copy()
fold_rows = []
for kind in KINDS:
    for trt in TREATMENTS:
        col = f'{kind}__{trt}'
        oof[col] = np.nan
        for k in folds:
            m = fold_arr == k
            p = np.load(fold_file(kind, trt, BEST[kind], k))
            assert len(p) == m.sum() and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
            oof.loc[m, col] = p
            mm = metrics(y[m], p)
            fold_rows.append(dict(model=kind, training=trt, fold=k, **mm))
assert oof[[f'{k}__{t}' for k in KINDS for t in TREATMENTS]].notna().all().all(), 'Some out-of-fold scores are missing'
assert oof['cas_key'].is_unique and len(oof) == len(dev)
fm = pd.DataFrame(fold_rows)
if ROUND2:        # keep the round-1 outputs as a record before they are overwritten
    for nm in ('fold_metrics_dev.csv', 'oof_scores_dev.parquet', 'oof_scores_dev.pkl', 'run_summary.json'):
        src, dst = SEARCH / nm, SEARCH / nm.replace('.', '_round1.', 1)
        if src.exists() and not dst.exists():
            dst.write_bytes(src.read_bytes())
fm.to_csv(SEARCH / 'fold_metrics_dev.csv', index=False)
try:
    oof.to_parquet(SEARCH / 'oof_scores_dev.parquet', index=False)
except Exception:
    oof.to_pickle(SEARCH / 'oof_scores_dev.pkl')

agg = fm.groupby(['model', 'training']).agg(ap_mean=('ap', 'mean'), ap_sd=('ap', 'std'), auc_mean=('auc', 'mean'), brier_mean=('brier', 'mean')).round(4)
agg['ap_pooled'] = [round(average_precision_score(y, oof[f'{a}__{b}']), 4) for a, b in agg.index]
agg['mean_pred'] = [round(oof[f'{a}__{b}'].mean(), 3) for a, b in agg.index]
print('Out-of-fold performance, nine combinations (descriptive; thresholds and selection come in notebook 05):')
display(agg)
print(f"Observed KSI rate {y.mean():.3f}. A mean predicted probability far above this is expected for weighted and oversampled fits.")
log = pd.read_csv(LOG)
print('\nCompute time in this log (hours):', round(log['t_total_min'].sum() / 60, 2), '| LR convergence warnings:', int(log.loc[log['kind'] == 'lr', 'conv_warnings'].sum()))
(SEARCH / 'run_summary.json').write_text(json.dumps(dict(finished=dt.datetime.now().isoformat(timespec='seconds'), sklearn=sklearn.__version__,
    best=BEST, folds_sha256=sha, rows=int(len(dev)), quick_test=QUICK_TEST, compute_hours=round(log['t_total_min'].sum() / 60, 2)), indent=1))
print('\nSaved to', SEARCH)
print('NEXT: notebook 05 (thresholds, one-standard-error model selection, freeze record). The 2025 labels stay sealed.')

Out-of-fold performance, nine combinations (descriptive; thresholds and selection come in notebook 05):


ap_mean   ap_sd  auc_mean  brier_mean  ap_pooled  mean_pred
model training                                                               
hgb   oversample   0.4534  0.0089    0.7496      0.2005     0.4531      0.437
      untreated    0.4558  0.0083    0.7502      0.1431     0.4556      0.211
      weight       0.4540  0.0089    0.7502      0.2010     0.4538      0.439
lr    oversample   0.4324  0.0109    0.7384      0.2083     0.4321      0.449
      untreated    0.4348  0.0110    0.7386      0.1456     0.4344      0.212
      weight       0.4328  0.0110    0.7385      0.2083     0.4325      0.449
rf    oversample   0.4456  0.0077    0.7449      0.1723     0.4454      0.368
      untreated    0.4486  0.0078    0.7460      0.1441     0.4484      0.213
      weight       0.4480  0.0086    0.7463      0.1734     0.4477      0.372

Observed KSI rate 0.212. A mean predicted probability far above this is expected for weighted and oversampled fits.

Compute time in this log (hours): 3.08 | LR convergence warnings: 0

Saved to /content/drive/MyDrive/stats19_severity/outputs/models/search
NEXT: notebook 05 (thresholds, one-standard-error model selection, freeze record). The 2025 labels stay sealed.
